## 1. Assess data coverage

The raw SEC XBRL files collected in NB01 contain financial information for different reporting periods and years. Before transforming the data, I first assess the coverage of each financial concept for every company.

For each JSON file, I extract the available annual observations, identify the first and last reporting years, and count the number of years available. This helps verify that the selected companies have sufficient and comparable data before building the final analysis dataset.

In [12]:
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

companies = {
    "MSFT": {"name": "Microsoft", "sector": "Technology"},
    "NVDA": {"name": "Nvidia", "sector": "Technology"},
    "AAPL": {"name": "Apple", "sector": "Technology"},
    "PFE": {"name": "Pfizer", "sector": "Healthcare"},
    "JNJ": {"name": "Johnson & Johnson", "sector": "Healthcare"},
    "SYK": {"name": "Stryker", "sector": "Healthcare"},
    "XOM": {"name": "ExxonMobil", "sector": "Energy"},
    "CVX": {"name": "Chevron", "sector": "Energy"},
    "DUK": {"name": "Duke Energy", "sector": "Energy"},
}

In [13]:
records = []

for path in sorted(RAW_DIR.glob("*.json")):
    ticker, tag = path.stem.split("_", 1)

    with open(path) as f:
        data = json.load(f)

    for obs in data["units"]["USD"]:
        if "start" not in obs:
            continue
        records.append({
            "ticker": ticker,
            "tag": tag,
            "start": obs["start"],
            "end": obs["end"],
            "val": obs["val"],
            "filed": obs["filed"],
        })

long = pd.DataFrame(records)
print(len(long), "observations read")

8413 observations read


In [14]:
# fy and fp describe the filing, not the period covered, so measure the period directly
long["days"] = (pd.to_datetime(long["end"]) - pd.to_datetime(long["start"])).dt.days
long = long[long["days"].between(350, 380)]

# Label each period by the calendar year it mostly covers. Taking the year from the
# end date misassigns 52/53-week fiscal years (J&J's FY2020 ended 3 Jan 2021) and
# shifts January year-ends forward (Nvidia's FY2026 covers most of calendar 2025).
start = pd.to_datetime(long["start"])
end = pd.to_datetime(long["end"])
long["year"] = (start + (end - start) / 2).dt.year

# companies restate figures, so the same year appears in several filings
long = long.sort_values("filed").groupby(["ticker", "tag", "year"], as_index=False).last()

print(len(long), "annual observations")

779 annual observations


In [15]:
revenue_tags = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
    "Revenues",
]

rev = long[long["tag"].isin(revenue_tags)]
overlap = rev.groupby(["ticker", "year"]).filter(lambda g: len(g) > 1)

overlap[["ticker", "year", "tag", "val"]].head(20)

,ticker,year,tag,val
69,AAPL,2017,RevenueFromContractWithCustomerExcludingAssess...,229234000000
70,AAPL,2018,RevenueFromContractWithCustomerExcludingAssess...,265595000000
79,AAPL,2017,Revenues,229234000000
80,AAPL,2018,Revenues,265595000000
156,CVX,2018,RevenueFromContractWithCustomerExcludingAssess...,158902000000
157,CVX,2019,RevenueFromContractWithCustomerExcludingAssess...,139865000000
158,CVX,2020,RevenueFromContractWithCustomerExcludingAssess...,94471000000
159,CVX,2021,RevenueFromContractWithCustomerExcludingAssess...,155606000000
160,CVX,2022,RevenueFromContractWithCustomerExcludingAssess...,235717000000
161,CVX,2023,RevenueFromContractWithCustomerExcludingAssess...,196913000000


In [16]:
# Each company uses a single revenue tag across its whole series, chosen by coverage.
# Mixing tags mid-series would create artificial steps: the contract-with-customer
# tags cover a subset of total revenue (excluding alliance revenue, royalties, and
# other operating income), so values differ from Revenues for CVX, PFE and XOM.
revenue_choice = {
    "MSFT": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "AAPL": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "JNJ":  "RevenueFromContractWithCustomerExcludingAssessedTax",
    "DUK":  "RevenueFromContractWithCustomerIncludingAssessedTax",
    "NVDA": "Revenues",
    "SYK":  "Revenues",
    "XOM":  "Revenues",
    "CVX":  "Revenues",
    "PFE":  "Revenues",   # only 2020 onward; contract tag stops in 2023
}

is_revenue = long["tag"].isin(revenue_tags)
chosen = long["ticker"].map(revenue_choice) == long["tag"]
tidy = long[~is_revenue | chosen].copy()

tidy.loc[tidy["tag"].isin(revenue_tags), "tag"] = "Revenue"

panel = tidy.pivot(index=["ticker", "year"], columns="tag", values="val").reset_index()
panel.columns.name = None

panel["sector"] = panel["ticker"].map(lambda t: companies[t]["sector"])
panel["name"] = panel["ticker"].map(lambda t: companies[t]["name"])

panel.head(10)

,ticker,year,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,PaymentsToAcquirePropertyPlantAndEquipment,ResearchAndDevelopmentExpense,Revenue,sector,name
0,AAPL,2007,5.470000e+09,3.495000e+09,NaN,7.820000e+08,NaN,Technology,Apple
1,AAPL,2008,9.596000e+09,6.119000e+09,NaN,1.109000e+09,NaN,Technology,Apple
2,AAPL,2009,1.015900e+10,8.235000e+09,NaN,1.333000e+09,NaN,Technology,Apple
3,AAPL,2010,1.859500e+10,1.401300e+10,NaN,1.782000e+09,NaN,Technology,Apple
4,AAPL,2011,3.752900e+10,2.592200e+10,NaN,2.429000e+09,NaN,Technology,Apple
5,AAPL,2012,5.085600e+10,4.173300e+10,NaN,3.381000e+09,NaN,Technology,Apple
6,AAPL,2013,5.366600e+10,3.703700e+10,8.165000e+09,4.475000e+09,NaN,Technology,Apple
7,AAPL,2014,NaN,3.951000e+10,9.571000e+09,6.041000e+09,NaN,Technology,Apple
8,AAPL,2015,8.126600e+10,5.339400e+10,1.124700e+10,8.067000e+09,NaN,Technology,Apple
9,AAPL,2016,6.623100e+10,4.568700e+10,1.273400e+10,1.004500e+10,NaN,Technology,Apple


In [25]:
panel = panel[panel["year"].between(2018, 2025)]

# how many companies have a value for each column, by year?
panel.set_index(["year", "ticker"]).notna().groupby("year").sum()

,name,sector,revenue,rd,net_income,ocf,capex,rd_intensity,net_margin,fcf,fcf_margin,capex_intensity
year,,,,,,,,,,,,
2018,9,9,7,8,9,9,8,6,7,8,7,7
2019,9,9,8,8,9,9,8,7,8,8,7,7
2020,9,9,9,8,9,9,8,8,9,8,8,8
2021,9,9,9,8,9,9,9,8,9,9,9,9
2022,9,9,9,8,9,9,9,8,9,9,9,9
2023,9,9,9,8,9,9,9,8,9,9,9,9
2024,9,9,9,8,9,9,9,8,9,9,9,9


In [27]:
expected = pd.MultiIndex.from_product(
    [companies.keys(), range(2018, 2025)], names=["ticker", "year"]
)

check = panel.set_index(["ticker", "year"]).reindex(expected)

value_cols = ["revenue", "rd", "net_income", "ocf", "capex"]

missing = check[value_cols].isna().stack()
missing[missing].reset_index().rename(columns={"level_2": "column"})[["ticker", "year", "column"]]

,ticker,year,column
0,NVDA,2018,revenue
1,NVDA,2018,capex
2,NVDA,2019,capex
3,NVDA,2020,capex
4,PFE,2018,revenue
5,PFE,2019,revenue
6,DUK,2018,rd
7,DUK,2019,rd
8,DUK,2020,rd
9,DUK,2021,rd


In [28]:
panel = panel.rename(columns={
    "ResearchAndDevelopmentExpense": "rd",
    "NetIncomeLoss": "net_income",
    "NetCashProvidedByUsedInOperatingActivities": "ocf",
    "PaymentsToAcquirePropertyPlantAndEquipment": "capex",
    "Revenue": "revenue",
})

panel["rd_intensity"] = panel["rd"] / panel["revenue"]
panel["net_margin"] = panel["net_income"] / panel["revenue"]
panel["fcf"] = panel["ocf"] - panel["capex"]
panel["fcf_margin"] = panel["fcf"] / panel["revenue"]
panel["capex_intensity"] = panel["capex"] / panel["revenue"]

panel = panel[[
    "ticker", "name", "sector", "year",
    "revenue", "rd", "net_income", "ocf", "capex",
    "rd_intensity", "net_margin", "fcf", "fcf_margin", "capex_intensity",
]].sort_values(["sector", "ticker", "year"])

panel.to_csv("../data/processed/company_financials.csv", index=False)
print(panel.shape)
panel.head()

(63, 14)


,ticker,name,sector,year,revenue,rd,net_income,ocf,capex,rd_intensity,net_margin,fcf,fcf_margin,capex_intensity
30,CVX,Chevron,Energy,2018,1.663390e+11,453000000.0,1.482400e+10,3.061800e+10,1.379200e+10,0.002723,0.089119,1.682600e+10,0.101155,0.082915
31,CVX,Chevron,Energy,2019,1.465160e+11,500000000.0,2.924000e+09,2.731400e+10,1.411600e+10,0.003413,0.019957,1.319800e+10,0.090079,0.096344
32,CVX,Chevron,Energy,2020,9.469200e+10,435000000.0,-5.543000e+09,1.057700e+10,8.922000e+09,0.004594,-0.058537,1.655000e+09,0.017478,0.094221
33,CVX,Chevron,Energy,2021,1.624650e+11,268000000.0,1.560000e+10,2.918700e+10,8.056000e+09,0.001650,0.096021,2.113100e+10,0.130065,0.049586
34,CVX,Chevron,Energy,2022,2.462520e+11,268000000.0,3.550000e+10,4.960200e+10,1.197400e+10,0.001088,0.144161,3.762800e+10,0.152803,0.048625


In [29]:
pd.read_csv("../data/processed/company_financials.csv").shape

(63, 14)

In [30]:
money_cols = ["revenue", "rd", "net_income", "ocf", "capex", "fcf"]

for col in money_cols:
    panel[col] = panel[col] / 1e9

panel = panel.rename(columns={c: f"{c}_bn" for c in money_cols})

panel = panel[[
    "ticker", "name", "sector", "year",
    "revenue_bn", "rd_bn", "net_income_bn", "ocf_bn", "capex_bn", "fcf_bn",
    "rd_intensity", "net_margin", "fcf_margin", "capex_intensity",
]].sort_values(["sector", "ticker", "year"])

panel.to_csv("../data/processed/company_financials.csv", index=False)
panel.head()

,ticker,name,sector,year,revenue_bn,rd_bn,net_income_bn,ocf_bn,capex_bn,fcf_bn,rd_intensity,net_margin,fcf_margin,capex_intensity
30,CVX,Chevron,Energy,2018,166.339,0.453,14.824,30.618,13.792,16.826,0.002723,0.089119,0.101155,0.082915
31,CVX,Chevron,Energy,2019,146.516,0.500,2.924,27.314,14.116,13.198,0.003413,0.019957,0.090079,0.096344
32,CVX,Chevron,Energy,2020,94.692,0.435,-5.543,10.577,8.922,1.655,0.004594,-0.058537,0.017478,0.094221
33,CVX,Chevron,Energy,2021,162.465,0.268,15.600,29.187,8.056,21.131,0.001650,0.096021,0.130065,0.049586
34,CVX,Chevron,Energy,2022,246.252,0.268,35.500,49.602,11.974,37.628,0.001088,0.144161,0.152803,0.048625
